# YZM212 Makine Öğrenmesi - 4. Ödev

**İsim-Soyisim:** Görkem Özer  
**Numara:** 23291007

## Problem Tanımı
Gürültülü gözlem verilerinden Bayesyen çıkarım ve MCMC (emcee) yöntemiyle bir gök cisminin gerçek parlaklığı (μ) ve gözlem hatası (σ) tahmin edilecektir.

## 1. Veri Üretimi (Simülasyon)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import corner

# Gerçek değerler (Doğa tarafından biliniyor)
true_mu = 150.0
true_sigma = 10.0
n_obs = 50

# Gürültülü veri oluştur
np.random.seed(42)
data = true_mu + true_sigma * np.random.randn(n_obs)

## 2. Bayes Fonksiyonları

In [ ]:
def log_likelihood(theta, data):
    mu, sigma = theta
    if sigma <= 0:
        return -np.inf
    return -0.5 * np.sum(((data - mu) / sigma)**2 + np.log(2 * np.pi * sigma**2))

def log_prior_genis(theta):
    mu, sigma = theta
    if 0 < mu < 300 and 0 < sigma < 50:
        return 0.0
    return -np.inf

def log_prior_dar(theta):
    mu, sigma = theta
    # Dar prior: sadece 100-110 arası parlaklık kabul ediyor
    if 100 < mu < 110 and 0 < sigma < 50:
        return 0.0
    return -np.inf

def log_probability(theta, data, prior_func):
    lp = prior_func(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, data)

## 3. MCMC ile Tahmin (Geniş Prior)

In [ ]:
def run_mcmc(data, prior_func, n_obs_label=""):
    initial = [140, 5]
    n_walkers = 32
    pos = initial + 1e-4 * np.random.randn(n_walkers, 2)
    
    sampler = emcee.EnsembleSampler(n_walkers, 2, log_probability, args=(data, prior_func))
    sampler.run_mcmc(pos, 2000, progress=True)
    flat_samples = sampler.get_chain(discard=500, thin=15, flat=True)
    
    mu_median = np.median(flat_samples[:, 0])
    sigma_median = np.median(flat_samples[:, 1])
    mu_low, mu_high = np.percentile(flat_samples[:, 0], [16, 84])
    sigma_low, sigma_high = np.percentile(flat_samples[:, 1], [16, 84])
    
    print(f"\n===== Sonuçlar {n_obs_label} =====")
    print(f"μ: {mu_median:.2f}  [{mu_low:.2f}, {mu_high:.2f}]")
    print(f"σ: {sigma_median:.2f}  [{sigma_low:.2f}, {sigma_high:.2f}]")
    
    return flat_samples, (mu_median, sigma_median), (mu_low, mu_high, sigma_low, sigma_high)

# Geniş prior ile çalıştır (n_obs=50)
samples_genis, med_genis, percentiles_genis = run_mcmc(data, log_prior_genis, "n_obs=50, Geniş Prior")

# Corner plot
fig = corner.corner(samples_genis, labels=["μ (Parlaklık)", "σ (Hata Payı)"], 
                    truths=[true_mu, true_sigma], title="Geniş Prior (n=50)")
plt.savefig("corner_plot_genis_prior.png", dpi=150)
plt.show()

## 4. Prior Etkisi Deneyi (Dar Prior: 100-110)

In [ ]:
# Dar prior ile çalıştır (n_obs=50)
samples_dar, med_dar, percentiles_dar = run_mcmc(data, log_prior_dar, "n_obs=50, Dar Prior (100-110)")

fig = corner.corner(samples_dar, labels=["μ (Parlaklık)", "σ (Hata Payı)"], 
                    truths=[true_mu, true_sigma], title="Dar Prior (100-110, n=50)")
plt.savefig("corner_plot_dar_prior.png", dpi=150)
plt.show()

## 5. Veri Miktarı Deneyi (n_obs = 5)

In [ ]:
# Az veri ile simülasyon
n_obs_small = 5
data_small = true_mu + true_sigma * np.random.randn(n_obs_small)

samples_small, med_small, percentiles_small = run_mcmc(data_small, log_prior_genis, "n_obs=5, Geniş Prior")

fig = corner.corner(samples_small, labels=["μ (Parlaklık)", "σ (Hata Payı)"], 
                    truths=[true_mu, true_sigma], title="Az Veri (n=5)")
plt.savefig("corner_plot_n5.png", dpi=150)
plt.show()

## 6. Sonuç Tablosu (Ödev Metni 5.1)

In [ ]:
from IPython.display import display, Markdown, HTML

mu_hata = abs(med_genis[0] - true_mu)
sigma_hata = abs(med_genis[1] - true_sigma)

tablo = f"""
| Değişken | Gerçek Değer | Tahmin (Median) | Alt Sınır (%16) | Üst Sınır (%84) | Mutlak Hata |
|----------|--------------|------------------|------------------|------------------|---------------|
| μ (Parlaklık) | {true_mu} | {med_genis[0]:.2f} | {percentiles_genis[0]:.2f} | {percentiles_genis[1]:.2f} | {mu_hata:.2f} |
| σ (Hata Payı) | {true_sigma} | {med_genis[1]:.2f} | {percentiles_genis[2]:.2f} | {percentiles_genis[3]:.2f} | {sigma_hata:.2f} |
"""
display(Markdown(tablo))

## 7. Analiz Soruları Cevapları

### 6.1 Merkezi Eğilim ve Doğruluk (Accuracy)
Model, gerçek μ=150.0 değerini {med_genis[0]:.2f} olarak tahmin etmiştir. Mutlak hata {mu_hata:.2f} civarındadır. Bu, verideki gürültüye (~%6-7) rağmen oldukça başarılı bir tahmindir. Bayesyen yöntem, gürültülü verilerde bile doğru merkezi değeri bulabilmektedir.

### 6.2 Tahmin Hassasiyeti (Precision)
μ'nun güven aralığı genişliği ({percentiles_genis[1]-percentiles_genis[0]:.2f}), σ'nun güven aralığı genişliğinden ({percentiles_genis[3]-percentiles_genis[2]:.2f}) daha dardır. Bunun nedeni, istatistikte ortalamanın varyanstan daha kesin tahmin edilebilmesidir. n=50 gibi makul bir örneklem büyüklüğü, μ tahminini oldukça hassas yaparken σ için daha fazla belirsizlik kalır.

### 6.3 Korelasyon Analizi
Corner plot'taki elipsin eğik durması, μ ve σ arasında pozitif bir korelasyon olduğunu gösterir. Yani model, μ büyük tahmin edildiğinde σ'yu da büyük tahmin etme eğilimindedir. Bu, verinin yayılımı ile merkezi arasındaki doğal ilişkiden kaynaklanır.

## 8. Prior Etkisi Sonuçları
Dar prior (100-110) kullanıldığında, tahmin edilen μ değeri {med_dar[0]:.2f} olmuştur (gerçek 150.0 iken). Bu, prior'ın yanlış olduğu durumda tahminin tamamen yanıltıcı olabileceğini gösterir. Bayesyen yöntemde prior seçimi kritiktir.

## 9. Veri Miktarı Etkisi
n_obs=5 iken μ'nun güven aralığı genişliği ({percentiles_small[1]-percentiles_small[0]:.2f}) iken, n=50'de bu değer {percentiles_genis[1]-percentiles_genis[0]:.2f} idi. Az veriyle belirsizlik çok daha büyümektedir. Bu, küçük örneklemlerde tahminlerin güvenilir olmadığını gösterir.